# Collaborative filtering - Matrix Factorisation with Surprise

In this notebook, you will learn another collaborative filtering technique called Matrix Factorisation, more specifically we will use `Singular Value Decomposition (SVD)` to factorise the **user-item** interaction matrix. 
The **benefit** of using matrix factorisation is that it can **capture** the **latent features** underlying the interactions between users and items. This allows us to make recommendations based on the user-item interaction matrix even if we have missing values.

We will use again  the Scikit-Surprise library to build a **SVD model** and make recommendations.

## Learning Objectives

At the end of this notebook, you should be able to:

- Build a matrix factorisation recommender with the Surprise `SVD` algorithm.
- Tune the SVD hyperparameters with cross-validated grid search.
- Evaluate rating predictions with RMSE and MAE.
- Recommend items to a new user by estimating their latent factor vector from a few ratings (the cold start problem).

## From similarity to latent factors

The sibling notebook on similarity-based collaborative filtering used a memory-based approach: it kept the full rating matrix in memory, found the users (or items) most similar to the target with a distance measure such as cosine or Pearson correlation, and averaged their ratings. That method is intuitive, but it stores every rating, struggles when the matrix is very sparse, and does not generalise beyond the neighbours it can see.

Matrix factorisation takes a model-based approach instead. Rather than comparing rows directly, it learns a compact model of the data: each user is described by a short vector of latent factors, and so is each item. A latent factor is a learned dimension of taste that is not labelled in advance. It might loosely correspond to how colourful a fish is or how peaceful its behaviour is, but the algorithm discovers these dimensions on its own by fitting the observed ratings. A predicted rating is then the dot product of a user vector and an item vector plus a few bias terms. Because the model is small and dense, it fills in the missing entries of the matrix and generalises to user-item pairs it has never seen, which is exactly what a recommender needs.

First we import the necessary libraries.

In [1]:
import pandas as pd
import numpy as np
from copy import copy
from surprise import Dataset
from surprise import Reader
from surprise import KNNWithMeans, SVD
from surprise.model_selection import GridSearchCV
from surprise.model_selection import train_test_split
from surprise import accuracy
from collections import defaultdict

Let's load our rating data. It contains the necessary `user_id`, `item_id` and the `rating` users gave to the fish items. Additionally it has some nice-to-have information about the fish items. There are 500 users with 300 rated fishes each. 

In [2]:
# Loading the dataset from the csv file
df = pd.read_csv("data/user_item_ratings.csv")
df.head(3)

,name,fish_group,visual_effect,user_id,rating,item_id
0,Bumblebee Platy - Xiphophorus maculatus,Tooth carp,especially colorful,0,6,492
1,King Tetra Super Blue DNZ - Inpaichthys kerri ...,Tetra,especially colorful,0,8,73
2,Black Scalar XL - Pterophyllum scalare,Cichlids,interesting body shape,0,8,347


The Scikit-Surprise library we want to use does not work with pandas DataFrames but with Dataset objects. So we need to create a Dataset object from our DataFrame. We also need to define the possible ratings with the Reader class.

In [3]:
# defines possible ratings
reader = Reader(rating_scale=(1, 10))
# Loads Pandas dataframe
data = Dataset.load_from_df(df[["user_id", "item_id", "rating"]], reader)

In order to validate our models we need to split our data into a trainset, which we will use to train our models. And a testset to validate the ability of our models to predict on unseen data. 

In [4]:
# Splitting the data into training and test set
trainset, testset = train_test_split(data, test_size=0.25, random_state=42)

With the data prepared, let us start modelling.

## Singular Value Decomposition

Another way to estimate the user ratings is **Singular Value Decomposition (SVD)**. **Singular value decomposition** is a matrix decomposition, where we decompose a given **m x n** matrix $A$ into three matrices **$U$**, $\Sigma$ and $V$ with the dimensions **m x r**, **r x r** and **n x r** respectively. Here r is usually small compared to m and n. And $\Sigma$ is non-zero only on the diagonal.

![](./images/SVD_USigmaV.png) 

By decomposing the matrix $A$ in this way, we drastically reduce the number of elements to be stored. Let $m = n = 100$ and $r = 5$ then the original matrix $A$ has $100 \times 100 = 10000$ elements. Whereas $U$ has $100 \times 5 = 500$, $\Sigma$ has $5$ (only diagonal elements) and $V$ again has $100 \times 5 = 500$ elements leading to a total of only $1005$ elements as opposed to the $10000$ elements of the original matrix $A$.

The rows of m of our user-item-rating matrix $A$ refer to the users and the n columns to the items (fishes in our case). The introduced **latent factors** (r in above example) can be interpreted as characteristics of the users for $U$ and item characteristics for $V$. So the latent factors are features of items or users, generated by our **SVD algorithm**. In our case they can tell us how much a user likes a certain `visual_effect` or how much a fish shows said `visual_effect`. $\Sigma$ then models the importance of each `visual_effect`.

The **actual implementation** of the algorithm in the **surprise library** will be outlined here to show an example of how to find an optimal decomposition. It became popular due to Simon Funk and his contribution to the Netflix competition. 

Here the rating that user u gives item i is denoted by $r_{ui}$ and will be estimated by:


$$
\hat r_{ui} = \mu + b_u + b_i + q^T_i p_u
$$

where 

- (u,i): **user-item pair**
- $\mu$: **average rating of all items**
- $b_i$: **average rating of item i minus $\mu$**
- $b_u$: **average rating given by user u minus $\mu$**
- $q_i$: **latent item factor vector**
- $p_u$: **latent user factor vector**
- $q^T_i p_u$: **dot product of the latent factors**

As shown in the formula above, the matrix decomposition is done by using only two matrices **$Q$** and **$P$**. The matrix **$Q$** contains the latent factors of the items and the matrix **$P$** contains the latent factors of the users. The latent factors and the biases are learned by minimising the regularised squared error:


$$
\sum_{r_{ui} \in R_{train}}(r_{ui} - \hat r_{ui})^2 + \lambda(b_i^2 + b_u^2 + ||q_i||^2 + ||p_u||^2)
$$

The minimization is performed by a simple stochastic gradient descent (parameters are updated iteratively):

$$
\begin{align*}
b_u &\leftarrow b_u + \gamma(e_{ui} - \lambda b_u)\\
b_i &\leftarrow b_i + \gamma(e_{ui} - \lambda b_i)\\
p_u &\leftarrow p_u + \gamma(e_{ui}*q_i - \lambda p_u)\\
q_i &\leftarrow q_i + \gamma(e_{ui}*p_u - \lambda q_i)
\end{align*}
$$

where $e_{ui} = r_{ui} - \hat r_{ui}$, $\gamma$ is the **learning rat**e and $\lambda$ is the **regularisation parameter**.

Now that we understand **SVD** let us train the algorithm on our train set.

In [5]:
# We'll use the famous SVD algorithm.
algo_svd = SVD(n_factors=100, n_epochs=20, lr_all=0.005, reg_all=0.02, random_state=42)

# Train the algorithm on the trainset, and
algo_svd.fit(trainset)

The **SVD** class takes a few parameters:
+ `n_factors`: The number of factors. Default is 100.
+ `n_epochs`: The number of iteration of the SGD procedure. Default is 20.
+ `lr_all`: The learning rate for all parameters. Default is 0.005.
+ `reg_all`: The regularisation term for all parameters. Default is 0.02.   
  

In [6]:
# Predict ratings for the testset
predictions = algo_svd.test(testset)

# Then compute RMSE
print(f"RMSE: {accuracy.rmse(predictions)}")

RMSE: 1.2240
RMSE: 1.223977727980435


With the default settings the SVD model scores an **RMSE of about 1.22** on the test set. That is close to the KNN model from the previous notebook (about 1.21), so on this dataset the two approaches perform similarly.

The advantage of SVD is that it stores compact latent factors for each user and item rather than a full item-item similarity matrix, which scales better to large catalogues.

**Note**: `.test()` is a method that evaluates the entire test set and returns the predictions as a list of `Prediction` objects. Each object details the `user ID`, `item ID`, `actual rating`, and `estimated rating`. Additionally, the `.predict()` method is used for predicting the rating for a single user-item pair, returning a `Prediction` object that includes the estimated rating among other details.

In [7]:
for element in predictions:
    print(
        f"user id:{element.uid}",
        f"item id:{element.iid}",
        f"estimated rating:{element.est}",
        f"real rating:{element.r_ui}",
    )

user id:116 item id:161 estimated rating:5.554108992989052 real rating:5.0
user id:98 item id:223 estimated rating:8.44075811867584 real rating:8.0
user id:15 item id:499 estimated rating:8.469358559049319 real rating:7.0
user id:140 item id:353 estimated rating:6.6794293426335445 real rating:8.0
user id:424 item id:381 estimated rating:3.0868024253807285 real rating:4.0
user id:204 item id:181 estimated rating:5.462807634561486 real rating:6.0
user id:438 item id:146 estimated rating:5.423294251523497 real rating:6.0
user id:411 item id:503 estimated rating:4.651153315209772 real rating:7.0
user id:245 item id:376 estimated rating:2.4943665309792813 real rating:1.0
user id:33 item id:213 estimated rating:7.50964121231037 real rating:7.0
user id:308 item id:420 estimated rating:5.736220598009222 real rating:8.0
user id:217 item id:131 estimated rating:5.551654466616441 real rating:4.0
user id:451 item id:371 estimated rating:8.926155660718855 real rating:9.0
user id:484 item id:97 esti

Let's have a look at the top 10 recommendations for a specific user. Though there is no implementation of this in surprise the documentation provides a function `get_top_n` that returns the top-N recommendations, if we provide the predictions of our model:

In [8]:
def get_top_n(predictions, n=10):
    """Return the top-N recommendation for each user from a set of predictions.

    Args:
    predictions(list of Prediction objects): The list of predictions, as
        returned by the test method of an algorithm.
    n(int): The number of recommendation to output for each user. Default
        is 10.


    Returns:
    A dict where keys are user (raw) ids and values are lists of tuples:
        [(raw item id, rating estimation), ...] of
        size n.
    """

    # First map the predictions to each user.
    top_n = defaultdict(list)

    for user_id, item_id, actual_rating, estimated_rating, _ in predictions:
        top_n[user_id].append((item_id, estimated_rating))

    # Then sort the predictions for each user and retrieve the k highest ones.
    for user_id, estimated_ratings in top_n.items():
        estimated_ratings.sort(
            key=lambda x: x[1], reverse=True
        )  # sort by rating estimation, descending. x[1] is the estimated rating.
        top_n[user_id] = estimated_ratings[:n]

    return top_n

What we will get is a list of ten tuples (item_id, estimated_rating). 

In [9]:
# Getting the top 10 recommendations for each user
top_10 = get_top_n(predictions, n=10)
top_10

defaultdict(list,
            {116: [(112, np.float64(9.52709043464313)),
              (140, np.float64(9.253994566057077)),
              (89, np.float64(9.18270843704729)),
              (144, np.float64(9.002098853081833)),
              (137, np.float64(8.88260257966253)),
              (195, np.float64(8.752453495100495)),
              (187, np.float64(8.746653776248962)),
              (120, np.float64(8.672840479681698)),
              (141, np.float64(8.554027408272168)),
              (136, np.float64(8.436979652545622))],
             98: [(223, np.float64(8.44075811867584)),
              (302, np.float64(8.356253157541992)),
              (193, np.float64(8.350195226067328)),
              (424, np.float64(8.101345756852021)),
              (119, np.float64(7.903426914990833)),
              (87, np.float64(7.879711745988386)),
              (447, np.float64(7.872085939291856)),
              (467, np.float64(7.866991967299512)),
              (437, np.float64(7.829582505

In [10]:
# Print the recommended items for a specific user
user_id = 201  # user id
# 10 best rated items for user id
top_10[user_id]

[(295, np.float64(9.397829664953495)),
 (223, np.float64(9.138156898661485)),
 (320, np.float64(8.019800246341706)),
 (299, np.float64(7.961246022754898)),
 (420, np.float64(7.728336611879998)),
 (169, np.float64(7.564070649343475)),
 (354, np.float64(7.294788851795303)),
 (426, np.float64(7.257307324062896)),
 (297, np.float64(7.169382286455697)),
 (132, np.float64(7.162808630836287))]

For user 201 the SVD model predicts item 295 highest (about 9.4), then item 223 (about 9.14), down to roughly 7.2 for the tenth item. The exact ordering differs from the KNN model because SVD ranks items by learned latent factors rather than by item-item similarity.

Let's make list of the top 10 item id's `top_iids`. And use it with the original fishes dataframe to get some characteristics of our recommended fishes. Apparently our user liked especially colorful fishes the most :).

In [11]:
# The top 10 recommendations for user_id 201 are:
top_items_id_user_id = []
for item_id, estimated_rating in top_10[user_id]:
    print(f"item id: {item_id}, estimated rating: {estimated_rating}")
    top_items_id_user_id.append(item_id)

item id: 295, estimated rating: 9.397829664953495
item id: 223, estimated rating: 9.138156898661485
item id: 320, estimated rating: 8.019800246341706
item id: 299, estimated rating: 7.961246022754898
item id: 420, estimated rating: 7.728336611879998
item id: 169, estimated rating: 7.564070649343475
item id: 354, estimated rating: 7.294788851795303
item id: 426, estimated rating: 7.257307324062896
item id: 297, estimated rating: 7.169382286455697
item id: 132, estimated rating: 7.162808630836287


In [12]:
top_items_id_user_id

[295, 223, 320, 299, 420, 169, 354, 426, 297, 132]

In [13]:
# Getting the name of the recommended items
recommended_fishes = (
    df.set_index("item_id")
    .loc[top_items_id_user_id][["name", "fish_group", "visual_effect"]]
    .drop_duplicates()
    .copy()
)
recommended_fishes

,name,fish_group,visual_effect
item_id,,,
295,2x Scalar Zebra - Pterophyllum scalare - DNZ,Cichlids,interesting body shape
223,Purple Fin Loach - Pseudogastromyzon laticeps,Loaches,interesting body shape
320,True Altum Scalar - Pterophyllum altum,Cichlids,forms territories (when spawning)
299,Four-striped Slim Cichlid - Julidochromis rega...,Cichlids,forms territories (when spawning)
420,Mosaic Gourami gold DNZ - Trichogaster leeri,Labyrinth fish,forms territories (when spawning)
169,Mouthbrooding witch catfish - Loricaria simillima,Catfish,forms territories (when spawning)
354,2x chestnut cichlid - Aequidens maronii,Cichlids,forms territories (when spawning)
426,2x peaceful fighting fish - Betta imbellis - pair,Labyrinth fish,forms territories (when spawning)
297,"2x scalar ""mix"" - Pterophyllum scalare - DNZ",Cichlids,forms territories (when spawning)


## Parameter Grid Search

The SVD algorithm in the surprise library comes with a lot of hyper parameters which influence the performance of the model. To search for optimal hyper parameters in machine learning Grid search is used. It performs an exhaustive search over a prior defined parameter space using cross-validation (hence the CV suffix). That means it will evaluate all of the possible parameter combinations of the search space in order to find and return the best combination.

This task, however, starts to become very time-consuming if there are many hyperparameters and the search space is huge. As you can see for cv= 3 (number of folds that will be evaluated) and for 3 parameters with 2 values, thus 8 combinations, the GridSearchCV runs 24 modelling steps in order to just come up with the best values for the three parameters.

Please feel free to try out different values and beat the default ones from our prediction above ;) and if you still have time improve your result even further!

In [14]:
param_grid = {
    "n_epochs": [
        10,
        20,
    ],  # The number of iteration of the SGD procedure. Default is 20.
    "lr_all": [0.002, 0.005],  # The learning rate for all parameters. Default is 0.005.
    "reg_all": [
        0.02,
        0.04,
    ],  # The regularization term for all parameters. Default is 0.02.
}
gs = GridSearchCV(SVD, param_grid, measures=["rmse", "mae"], cv=3)

gs.fit(data)

In [15]:
print("best RMSE:")
print(gs.best_score["rmse"])
print("best parameters:")
print(gs.best_params["rmse"])
print("best MAE:")
print(gs.best_score["mae"])
print("best parameters:")
print(gs.best_params["mae"])
print("best estimator:")
print(gs.best_estimator)

best RMSE:
1.192694129120037
best parameters:
{'n_epochs': 10, 'lr_all': 0.005, 'reg_all': 0.04}
best MAE:
1.0254199938789803
best parameters:
{'n_epochs': 10, 'lr_all': 0.005, 'reg_all': 0.02}
best estimator:
{'rmse': <surprise.prediction_algorithms.matrix_factorization.SVD object at 0x000001D9C748FBD0>, 'mae': <surprise.prediction_algorithms.matrix_factorization.SVD object at 0x000001D9C748FAC0>}


Grid search evaluates every combination in the parameter grid with 3-fold cross-validation and reports the best one. Here it finds a best cross-validated RMSE of roughly 1.19, a small improvement over the default model.

The exact best score and the winning parameters vary a little between runs, because the cross-validation folds are not seeded. Run it a few times and you will see the numbers move slightly.

In [16]:
# Get the best model
best_svd = gs.best_estimator["mae"]
# Since GridSearchCV does not train the best_estimator on full data automatically, we need to retrain it:
best_svd.fit(trainset)

print(f"{accuracy.mae(predictions)}")

MAE:  1.0413
1.041289742149229


This prints the MAE of the SVD predictions, about 1.04. MAE is the average absolute error. Unlike RMSE it does not square the errors, so it is easier to read directly: on average the predicted rating is about one point away from the true rating on the 1 to 10 scale.

Keep in mind that getting the best score in recommender systems is not always the best for your business. Very accurate scores can lead to recommendations perceived as boring by the user, because the items are too similar to items they already know. So always keep the user in mind when optimising your model.

## Predictions for a new user
Let's imagine we have a new user who has not rated any fish yet. This is a common issue called the **cold start problem**. For this user we could use the **most popular** fishes as a recommendation. This is a simple but effective way to start with. We can also ask the user to rate some items and then use the **user-based** or **item-based** collaborative filtering to make recommendations. One could directly use the trained model to make predictions for the new user or retrain the model with the new user's ratings.

In [17]:
df

,name,fish_group,visual_effect,user_id,rating,item_id
0,Bumblebee Platy - Xiphophorus maculatus,Tooth carp,especially colorful,0,6,492
1,King Tetra Super Blue DNZ - Inpaichthys kerri ...,Tetra,especially colorful,0,8,73
2,Black Scalar XL - Pterophyllum scalare,Cichlids,interesting body shape,0,8,347
3,Red Line Algae Eater - Sahyadria (Puntius) den...,Barbs,especially colorful,0,8,86
4,"Platy ""Red Wagtail"" - Xiphophorus maculatus",Tooth carp,especially colorful,0,5,455
...,...,...,...,...,...,...
149995,"2x Gourami ""Gold"" - Colisa chuna - pair",Labyrinth fish,especially colorful,499,6,397
149996,Tortoiseshell Catfish - LDA 16 - Ancistrus sp....,Catfish,forms precincts,499,5,236
149997,L184 / L107 Brilliant catfish - Ancistrus sp. ...,Catfish,especially colorful,499,6,176
149998,Yellow Congo Tetra - Alestopetersius caudalis,Tetra,interesting body shape,499,2,72


We will follow the approach of asking the new user to rate some items and then use the latent factors of the SVD model to make recommendations according to the equation:

$$
\hat r_{ui} = \mu + b_u + b_i + q^T_i p_u
$$

First, we will need to estimate the user latent feature matrix **p** based on the new user's ratings and the item latent feature matrix **q** from the model. We can then use the equation above to predict the ratings for the new user. We will then recommend the top 10 items with the highest predicted ratings.

### Collect Ratings from New User

In [18]:
new_user_ratings = [
    {"user_id": 500, "item_id": 1, "rating": 10},
    {"user_id": 500, "item_id": 2, "rating": 9},
    {"user_id": 500, "item_id": 3, "rating": 8},
    {"user_id": 500, "item_id": 40, "rating": 7},
    {"user_id": 500, "item_id": 5, "rating": 6},
    {"user_id": 500, "item_id": 6, "rating": 5},
    {"user_id": 500, "item_id": 390, "rating": 4},
    {"user_id": 500, "item_id": 8, "rating": 3},
    {"user_id": 500, "item_id": 9, "rating": 2},
    {"user_id": 500, "item_id": 10, "rating": 1},
]

In [19]:
# Create a new dataframe with the new user ratings
new_user_ratings_df = pd.DataFrame(new_user_ratings)
new_user_ratings_df

,user_id,item_id,rating
0,500,1,10
1,500,2,9
2,500,3,8
3,500,40,7
4,500,5,6
5,500,6,5
6,500,390,4
7,500,8,3
8,500,9,2
9,500,10,1


### Estimate the New User's Latent Feature vector P
We need to estimate the user’s latent feature vector that best explains the new user's provided ratings, using the known item factors from the SVD model.

In [20]:
# Items that the new user has rated
item_ids = new_user_ratings_df["item_id"].values
item_ids

array([  1,   2,   3,  40,   5,   6, 390,   8,   9,  10])

#### Extract the item factors q for the rated items from the trained model

In [21]:
item_factors = []
for item_id in item_ids:
    # Convert the raw id to inner id, which is in the same order of the items  as in the model
    item_idx = algo_svd.trainset.to_inner_iid(item_id)
    # print(item_idx)
    # Get the item factor
    item_factor = algo_svd.qi[item_idx]
    # print(item_factor)
    item_factors.append(item_factor)

# Convert the list of item factors to a numpy array
Q = np.array(item_factors)
# Check the shape of the array
Q.shape

(10, 100)

As we can see, `Q` is a numpy array with the shape `(number of rated items, number of latent factors)`, here `(10, 100)`.

#### Extract the item biases b for the rated items from the trained model

In [22]:
item_biases = []
for item_id in item_ids:
    # Convert the raw id to inner id, which is in the same order of the items  as in the model
    item_idx = algo_svd.trainset.to_inner_iid(item_id)
    item_bias = algo_svd.bi[item_idx]
    item_biases.append(item_bias)

b = np.array(item_biases)
b.shape

(10,)

#### Compute the user bias as the average of all user biases

In [23]:
# Compute the user bias as the average of all user biases
user_bias = np.mean(algo_svd.bu)
user_bias

np.float64(-0.0019511132548782585)

#### Compute the user latent feature vector P
To estimate the user latent feature vector, we need to solve the following equation:

$$
\hat r_{ui} = \mu + b_u + b_i + q^T_i p_u
$$

For each rated item, we have the following equation:

$$
r_{ui} = \mu + b_u + b_i + q^T_i p_u
$$

We can rewrite this equation as follows:

$$
r_{ui} - \mu - b_i - b_u = q^T_i p_u
$$

We can then stack these equations for all rated items to form a matrix equation:

$$
R - \mu - B - B_u = Q P_u
$$

where:
- $R$ is the vector of all ratings for the rated items
- $B$ is the vector of all item biases for the rated items
- $B_u$ is the vector of all user biases for the rated items
- $Q$ is the matrix of all item latent feature vectors for the rated items
- $P_u$ is the user latent feature vector we want to estimate
- $\mu$ is the average rating of all items


We can solve this equation for $P_u$ using the following equation:

$$
P_u = (Q^T Q + \lambda I)^{-1} Q^T (R - \mu - B -B_u)

$$

where $\lambda$ is the regularisation parameter and $I$ is the identity matrix.



In [24]:
actual_ratings = new_user_ratings_df["rating"].values
actual_ratings

array([10,  9,  8,  7,  6,  5,  4,  3,  2,  1])

In [25]:
# global average rating
mu = algo_svd.trainset.global_mean
mu

np.float64(5.557253333333334)

In [26]:
adjusted_ratings = actual_ratings - mu - b - user_bias
adjusted_ratings

array([ 4.35018601,  3.45469711,  2.28513657,  1.27993198,  0.46505545,
       -0.82081641, -1.40478252, -2.50576588, -3.58946986, -4.47711758])

In [27]:
# Compute Q^T Q
QTQ = np.dot(Q.T, Q)
QTQ.shape

(100, 100)

In [28]:
# # Identity matrix of size equal to the number of features
I = np.eye(Q.shape[1])
I.shape

(100, 100)

In [29]:
# Regularized matrix (Compute QᵀQ + λI)
lambda_reg = 0.1
regularized_matrix = QTQ + lambda_reg * I

In [30]:
# Inverse of the regularized matrix (Compute (QᵀQ + λI)⁻¹)
inv_regularized_matrix = np.linalg.inv(regularized_matrix)
inv_regularized_matrix.shape

(100, 100)

In [31]:
# Compute final P_u = (QᵀQ + λI)⁻¹ Qᵀ (R - μ - B - B_u)
P_u = inv_regularized_matrix.dot(Q.T).dot(adjusted_ratings)
P_u

array([-0.05404421,  0.29668827,  0.15580165, -0.30100544, -0.53904404,
       -0.13045565, -0.53747089,  0.2160465 , -1.06302703,  0.79989405,
        0.10579009,  0.09478285, -0.0366495 ,  0.20684936,  0.1375818 ,
       -0.24025583,  0.55655734, -0.42047448,  0.00620791, -0.17403912,
        1.21214961,  0.17152483, -0.35367131,  0.0457491 ,  0.21606482,
       -0.28715897, -0.13394288,  0.35368158, -0.37069357,  0.08623157,
        0.36251989, -0.4688531 ,  0.15227817,  0.22267022,  0.88016265,
       -0.46236312,  0.54110148,  0.45474542, -0.0940569 , -0.0761789 ,
       -0.42932054,  0.2711499 , -0.56866047,  0.02767676, -0.09961019,
        0.09694482, -0.45345576,  0.07784663,  0.08527814, -0.07045536,
       -0.07985084,  0.1720782 , -0.39367865, -0.11955453,  0.57148018,
        0.01704219,  0.15209131,  0.94522383,  0.55567938,  0.03139893,
        1.09298318, -0.36035317,  0.31545258, -0.2021788 , -0.55494767,
       -0.00417885,  0.53553708,  0.91420067,  0.33190409, -0.30

In [32]:
estimated_rating = mu + user_bias + b + np.dot(Q, P_u)
estimated_rating

array([9.85037159, 8.92364889, 7.95424483, 6.99144295, 5.99681185,
       5.03944467, 4.01557659, 3.07740261, 2.09224449, 1.09693794])

### Make Recommendations for the New User on Items Not Yet Rated
Now that we have estimated the new user's latent feature vector, we can use it to predict the ratings for the items not yet rated by the user. We can then recommend the top 10 items with the highest predicted ratings.

#### Get all items

In [33]:
all_items = df["item_id"].unique()
all_items

array([492,  73, 347,  86, 455,  77, 446, 529,  89, 208,   6, 249, 405,
        75, 247, 523, 479, 101, 155,  55, 234, 177, 158, 331, 209,   2,
       338, 353,  10, 374, 117, 173, 478, 274, 530, 467, 289,  72, 342,
       265, 149, 354, 518, 468, 296,  79, 132, 238, 380,  30, 231, 505,
       438,   9, 525, 176, 104, 211, 140, 148, 196, 312,  76, 532,  70,
       481, 469, 364, 369, 255,  78, 218, 448, 489,  68, 497,  63, 452,
       433,  82, 341,  33,  84,  90,   0,  11, 407,  22, 352, 365, 470,
       172, 153, 367,  18, 195, 411,  15, 293,  39, 493, 524,  46,  93,
       323, 321, 305, 536, 506, 192, 348, 180, 301, 137, 441, 325, 268,
       533, 244, 426,  69, 377, 318, 185, 131, 154, 228, 464, 124, 375,
       447, 424, 182, 355, 290, 472, 227, 245, 421,  19, 494,  56, 396,
       483, 362, 461, 316,  25, 275,  42, 333, 225, 184, 302, 126, 272,
       357,  31, 113, 210, 382, 167,  57, 229,  24,  17, 437,  66, 304,
       261, 248, 376,  94, 408, 509,  23, 442, 284, 237,   5, 11

#### Get the items not rated by the new user

In [34]:
items_not_rated = np.setdiff1d(all_items, item_ids)
items_not_rated

array([  0,   4,   7,  11,  12,  13,  14,  15,  16,  17,  18,  19,  20,
        21,  22,  23,  24,  25,  26,  27,  28,  29,  30,  31,  32,  33,
        34,  35,  36,  37,  38,  39,  41,  42,  43,  44,  45,  46,  47,
        48,  49,  50,  51,  52,  53,  54,  55,  56,  57,  58,  59,  60,
        61,  62,  63,  64,  65,  66,  67,  68,  69,  70,  71,  72,  73,
        74,  75,  76,  77,  78,  79,  80,  81,  82,  83,  84,  85,  86,
        87,  88,  89,  90,  91,  92,  93,  94,  95,  96,  97,  98,  99,
       100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112,
       113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125,
       126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138,
       139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151,
       152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164,
       165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177,
       178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 18

The numpy function `np.setdiff1d` is a very efficient way to find the difference between two arrays.

#### Extract the item factors for the items not rated by the new user from the trained model

In [35]:
item_factors_not_rated = []
for item_id in items_not_rated:
    # Convert the raw id to inner id, which is in the same order of the items as in the model
    item_idx = algo_svd.trainset.to_inner_iid(item_id)
    item_factor = algo_svd.qi[item_idx]
    item_factors_not_rated.append(item_factor)

In [36]:
# Convert the list to a numpy array
Q_not_rated = np.array(item_factors_not_rated)
Q_not_rated.shape

(527, 100)

#### Extract the item biases for the items not rated by the new user from the trained model

In [37]:
# Extract the item biases for the items not rated by the new user from the trained model
item_biases_not_rated = []
for item_id in items_not_rated:
    # Convert the raw id to inner id, which is in the same order of the items as in the model
    item_idx = algo_svd.trainset.to_inner_iid(item_id)
    item_bias = algo_svd.bi[item_idx]
    item_biases_not_rated.append(item_bias)

In [38]:
# Convert the list to a numpy array
b_not_rated = np.array(item_biases_not_rated)
b_not_rated

array([-0.00248718, -0.11030225, -0.09791175,  0.05359304, -0.03466117,
        0.10039517, -0.03551882,  0.16052157,  0.07239861, -0.24271321,
        0.1458897 ,  0.13147177, -0.25476038, -0.10364521,  0.2078165 ,
       -0.04670172,  0.10905252, -0.16867866, -0.08454912,  0.09280656,
       -0.18426193, -0.09417497, -0.00777216,  0.00616008,  0.03454433,
       -0.07389121, -0.05316588, -0.0842521 , -0.11783248,  0.13928161,
       -0.02440492,  0.06264585, -0.27723624, -0.04749771,  0.12383197,
       -0.04458918,  0.03940131,  0.14254198, -0.25293356, -0.15353608,
       -0.01852769,  0.01838471, -0.03609828, -0.02231032,  0.19933242,
       -0.07724093, -0.05973536,  0.07663156, -0.12104419,  0.0287802 ,
        0.17183762,  0.05685681,  0.03532065,  0.0927615 ,  0.10301763,
       -0.03988304, -0.05870264, -0.18286598, -0.00051613, -0.01545052,
       -0.17443443, -0.16414451, -0.12759886, -0.11419902,  0.06849067,
        0.01584327, -0.01558838, -0.14244462, -0.05417393, -0.04

#### Compute the estimated ratings for the items not rated by the new user

In [39]:
estimated_ratings_not_rated = mu + b_not_rated + user_bias + np.dot(Q_not_rated, P_u)
estimated_ratings_not_rated

array([5.26049746, 3.75304224, 3.57102172, 5.82175793, 6.10915557,
       6.68117428, 3.42496443, 6.17028827, 7.01857345, 2.70396787,
       6.72298525, 6.61739658, 4.08380637, 4.59956316, 6.97048223,
       3.90272034, 6.04433157, 4.35992863, 4.32045416, 8.01349469,
       5.7704863 , 3.329621  , 4.00289397, 3.90681043, 4.37937978,
       3.8112865 , 3.73234559, 3.86360383, 3.17254096, 4.93394805,
       6.02159043, 7.86609649, 3.16257812, 5.230185  , 7.35793095,
       3.62019327, 4.60004424, 5.87108104, 3.59645026, 3.70248035,
       5.99999873, 7.02230271, 5.40536355, 4.50892624, 6.7688378 ,
       3.57372823, 4.03839108, 5.39775673, 5.18174423, 5.70785891,
       6.25729482, 6.31930313, 5.81551135, 7.41679004, 5.17026165,
       5.22846474, 4.42576086, 3.09686986, 6.5075505 , 7.86711442,
       3.34444339, 5.29036145, 5.43236383, 6.80376576, 7.30996507,
       5.82941245, 6.24418088, 2.93675308, 4.49011182, 4.03239967,
       3.82358782, 5.44253641, 4.14847776, 7.64990401, 4.28796

#### Get the top 10 items with the highest predicted ratings

In [40]:
name_of_items_not_rated = (
    df.set_index("item_id").loc[items_not_rated]["name"].drop_duplicates().values
)
name_of_items_not_rated

<StringArray>
[                                'Cardinalfish - Tanichthys albonubes',
                                   'Neon Tetra - Paracheirodon innesi',
                        'Red-tailed Feather Tetra - Hemiodus gracilis',
                            'Sparkling Tetra - Hyphessobrycon amandae',
                   'Harlequin Barb - Trigonostigma heteromorpha - DNZ',
                               'Algae Eater - Crossocheilus siamensis',
             'Web Brush Algae Eater - Crossocheilus reticulatus - DNZ',
                                   'Copper Tetra DNZ - Hasemania nana',
 'Bitterling barb - Puntius (Barbus) titteya - DNZ / strong colouring',
             'Espes harlequin rasbora - Trigonostigma (Rasbora) espei',
 ...
                   'Golden Fringe Lipstick - Epalzeorhynchos frenatum',
                     'White-fronted Knifefish - Apteronotus albifrons',
                           'Finite bichir - Polypterus finiteri - 4cm',
                        'Axolotl "Albino" - A

In [41]:
# Create a dataframe with the name of the items and the estimated ratings for the items not rated by the new user
name_estimated_ratings_not_rated_df = pd.DataFrame(
    {
        "name": name_of_items_not_rated,
        "item_id": items_not_rated,
        "estimated_rating": estimated_ratings_not_rated,
    }
)
name_estimated_ratings_not_rated_df

,name,item_id,estimated_rating
0,Cardinalfish - Tanichthys albonubes,0,5.260497
1,Neon Tetra - Paracheirodon innesi,4,3.753042
2,Red-tailed Feather Tetra - Hemiodus gracilis,7,3.571022
3,Sparkling Tetra - Hyphessobrycon amandae,11,5.821758
4,Harlequin Barb - Trigonostigma heteromorpha - DNZ,12,6.109156
...,...,...,...
522,Dwarf Snakehead - Channa gachua blue bengal - ...,532,5.506796
523,Beautiful Snakehead - Channa pulchra - DNZ - 10cm,533,4.375074
524,Axolotl - Ambystoma mexicanum - DNZ,534,5.214459
525,Brown White-fronted Knifefish - Apteronotus al...,535,5.338067


In [42]:
# Get the name of the top 10 items
top_10_items = name_estimated_ratings_not_rated_df.sort_values(
    by="estimated_rating", ascending=False
).head(10)
top_10_items

,name,item_id,estimated_rating
373,Redtail Crested Puffer - Carinotetraodon irrub...,382,9.205385
332,Scalar koi red DNZ - Pterophyllum scalare,341,8.842338
336,Scalar koi brilliant DNZ - Pterophyllum scalare,345,8.787262
477,"Guppy male ""Dumbo ear"" - Poecilia reticulata",487,8.441871
397,Betta male Dragon metallic lg - Betta splendens,407,8.357021
357,Striped Garfish Gold - Aplocheilus lineatus,366,8.311934
140,Orange Laser Mailed Catfish - Corydoras spec. ...,149,8.257980
426,Male Lemon Guppy - Poecilia reticulata,436,8.228494
447,"Platy ""bleeding heart"" - Xiphophorus maculatus",457,8.195823
413,Female fighting fish super delta lg - Betta sp...,423,8.105941


For the new user the reconstructed top-10 are items such as 382, 341 and 345, with predicted ratings from about 9.2 down to 8.1. Because we estimated the user's latent vector from only ten ratings, these predictions are rougher than for a user with a full history, but they already reflect the pattern in the ratings the new user gave.

## Summary

In this notebook you:

- Built a matrix factorisation recommender with the Surprise `SVD` algorithm, decomposing the user-item matrix into latent factors.
- Evaluated the predictions and reached an RMSE of about 1.22 and an MAE of about 1.04 on the test set.
- Tuned the SVD hyperparameters with cross-validated grid search, which nudged the RMSE down to roughly 1.19.
- Recommended items to a brand new user by estimating their latent factor vector from a handful of ratings.

Remember that the most accurate model is not always the best for the business: very tight predictions can feel repetitive to users, so keep the user experience in mind when tuning.

## References & Further Reading

- [**Surprise Library**](https://surprise.readthedocs.io/en/stable/index.html): documentation for the scikit-surprise recommender library used here.
- [**SVD in Surprise**](https://surprise.readthedocs.io/en/stable/matrix_factorization.html#surprise.prediction_algorithms.matrix_factorization.SVD): the SVD algorithm, its parameters and the prediction formula.
- [**GridSearchCV**](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html): scikit-learn reference for exhaustive cross-validated hyperparameter search.
- [**Singular Value Decomposition**](https://en.wikipedia.org/wiki/Singular_value_decomposition): the linear algebra behind decomposing a matrix into U, sigma and V.
- [**Matrix Factorisation (recommender systems)**](https://en.wikipedia.org/wiki/Matrix_factorization_(recommender_systems)): overview of Funk MF, SVD++ and related recommender variants.
- [**Collaborative Filtering**](https://en.wikipedia.org/wiki/Collaborative_filtering): memory-based versus model-based approaches and their trade-offs.
- [**Netflix Prize**](https://en.wikipedia.org/wiki/Netflix_Prize): the competition that popularised matrix factorisation for recommendation.
- [**Simon Funk's SVD write-up**](https://sifter.org/simon/journal/20061211.html): the original blog post describing the gradient-descent SVD used in Surprise.
- [**Cold Start Problem**](https://en.wikipedia.org/wiki/Cold_start_(recommender_systems)): why new users and items are hard to recommend for, and how to mitigate it.